# Span-free information extraction with GLiNER 2.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/aibackends/blob/main/examples/notebooks/gliner25_information_extraction_colab.ipynb)

This notebook turns the capabilities in Fastino's [GLiNER 2.5 release post](https://fastino.ai/blog/gliner2-5-span-free-information-extraction) into runnable use cases. GLiNER 2.5 predicts entity boundaries instead of enumerating fixed-width spans, so it can represent any span that fits inside an encoded window and scan longer documents with overlapping chunks.

**What this notebook covers**

1. Long-document contract review with source-verifiable global offsets
2. Schema-shaped invoice records
3. Span-level sentiment attributes
4. Constrained model/agent routing
5. Typed joint entity-relation graphs for agent memory
6. Native batches and multilingual extraction

All inference runs locally. No API key or remote inference service is required.

## Setup

`gliner2[local]` installs the schema API plus PyTorch and Transformers for local inference. Version 2.0.0 or newer is required for the GLiNER 2.5 boundary checkpoints and advanced decoders.

In [ ]:
!pip install -q "gliner2[local]>=2.0.0"

# If an import fails after installation, use Runtime > Restart session,
# then continue from the next cell.

## Pick a checkpoint

| Checkpoint | Parameters | Best fit |
|---|---:|---|
| `gliner2.5-small-v1` | 74M | Fast English CPU/edge workloads |
| `gliner2.5-base-v1` | 194M | Default English multi-task quality |
| `gliner2.5-multi-v1` | 287M | Multilingual extraction and classification |

The notebook defaults to `small` so it stays quick on a free CPU runtime. Change one line to compare `base` or `multi`; the API is identical.

In [ ]:
import json
import time
import warnings

import torch
from gliner2 import AttributeGroup, AutoExtractor

warnings.filterwarnings(
    'ignore',
    message='Checkpoint uses legacy list-valued extra_special_tokens metadata',
)
warnings.filterwarnings(
    'ignore',
    message="Encoder rejected attn_implementation='sdpa'",
)

MODEL_ID = 'fastino/gliner2.5-small-v1'
# MODEL_ID = 'fastino/gliner2.5-base-v1'
# MODEL_ID = 'fastino/gliner2.5-multi-v1'

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

started = time.perf_counter()
model = AutoExtractor.from_pretrained(MODEL_ID, map_location=DEVICE)
print('model:', MODEL_ID)
print('architecture:', type(model).__name__)
print('device:', DEVICE)
print(f'load: {time.perf_counter() - started:.2f}s')

## 1. Contract review over a full document

`extract_entities_long` scans overlapping word chunks instead of silently truncating the document. Every returned `[start, end)` pair is remapped to the original text, so a downstream reviewer or redactor can verify the source exactly.

Boundary prediction removes the old fixed span-width ceiling, but a span must still fit wholly inside at least one chunk. Relations likewise require both endpoints in the same chunk.

In [ ]:
padding = ' '.join(
    [
        'This agreement contains commercial background, definitions, service levels, '
        'and reporting terms.'
    ]
    * 8
)
contract = (
    f'{padding} Northstar Analytics LLC signed this services agreement with '
    'Blue Harbor Bank on August 18, 2026. Either party may terminate this Agreement for '
    'convenience by giving the other party at least thirty (30) days\' prior written notice '
    f'delivered by certified mail. {padding}'
)

contract_result = model.extract_entities_long(
    contract,
    {
        'contract party': 'Organizations that are parties to the agreement',
        'effective date': 'The date the agreement starts',
        'termination clause': 'The complete clause describing how the agreement can end',
    },
    chunk_size=64,
    chunk_overlap=20,
    include_spans=True,
    include_confidence=True,
)

verified = 0
for entities in contract_result['entities'].values():
    for entity in entities:
        assert contract[entity['start']:entity['end']] == entity['text']
        verified += 1

print(f'{len(contract.split())} words scanned; {verified} global offsets verified')
print(json.dumps(contract_result, indent=2))

## 2. Schema-shaped records

`extract_json` maps free text into named records and typed fields. The schema controls output shape; this is useful for invoices, support tickets, clinical notes, and other ingestion pipelines.

In [ ]:
invoice_text = (
    'Invoice INV-2048 from Acme Labs totals $1,240.00 and is due '
    'September 30, 2026.'
)
invoice = model.extract_json(
    invoice_text,
    {
        'invoice': [
            'invoice_number::str',
            'vendor::str',
            'total::str',
            'due_date::str',
        ]
    },
)
print(json.dumps(invoice, indent=2))

## 3. Span attributes in the same pass

Attributes qualify each retained span rather than the whole document. Here one mixed review can attach different sentiment labels to different product mentions without a second per-entity classifier.

In [ ]:
feedback = 'The Atlas camera is excellent, but its battery life is disappointing.'
attribute_schema = (
    model.create_schema()
    .entities(['product'])
    .entity_attributes(
        {
            'sentiment': AttributeGroup(
                ['positive', 'negative', 'neutral'],
                applies_to=['product'],
                qualify_labels=True,
            )
        }
    )
)
feedback_result = model.extract(
    feedback,
    attribute_schema,
    include_spans=True,
    include_confidence=True,
)

for product in feedback_result['entities']['product']:
    assert feedback[product['start']:product['end']] == product['text']
    print(product['text'], '->', product['sentiment'])

## 4. Constrained model or agent routing

`classify_text` is enough for independent labels. Use `Classifier` when decisions must agree across tasks. The decoder below cannot return a `delete` route without the `delete` effect, or a `read` route with a destructive effect.

In [ ]:
from gliner2.classification import Classifier, ClassificationSchema
from gliner2.classification import constraints as C

classifier = Classifier(model)
routing_schema = (
    ClassificationSchema()
    .single('route', ['read', 'write', 'delete'])
    .multi(
        'effects',
        ['read_only', 'create', 'modify', 'delete'],
        min_labels=1,
        max_labels=2,
    )
    .constrain(
        C.implies(('route', 'delete'), ('effects', 'delete')),
        C.implies(('route', 'read'), ('effects', 'read_only')),
        C.excludes(('route', 'read'), ('effects', 'delete')),
        C.excludes(('route', 'read'), ('effects', 'modify')),
    )
)

route = classifier.classify('Delete the temporary file from /tmp.', routing_schema)
print('route:', route.value('route'))
print('effects:', route.selected('effects'))
print('feasible:', route.feasible)
print(json.dumps(route.to_dict(), indent=2))

## 5. Joint information extraction for agent memory

`JointIE` chooses entities and relations as one typed graph. Each edge points to an entity in the same result, `works_for` is constrained to person → organization, `located_in` to organization → location, and self-loops are forbidden.

In [ ]:
from gliner2.joint_ie import JointIE, JointIEConfig

joint = JointIE(model)
graph_schema = (
    joint.create_schema()
    .entities(['person', 'organization', 'location'])
    .relation('works_for', 'person', 'organization', unique_head=True)
    .relation('located_in', 'organization', 'location', unique_head=True)
    .no_self_loops()
)
graph = joint.extract(
    'Maya Chen leads Northstar Analytics in Singapore.',
    graph_schema,
    config=JointIEConfig(optimizer='beam', beam_size=16),
)

print('feasible:', graph.feasible)
for relation in graph.relations:
    head = graph.entity(relation.head)
    tail = graph.entity(relation.tail)
    print(f'{head.text} -{relation.type}-> {tail.text} ({relation.confidence:.2f})')
print(json.dumps(graph.to_dict(), indent=2))

## 6. Native batches and multilingual text

Batch methods reuse one schema and encoder call across several texts. For non-English production inputs, select `gliner2.5-multi-v1` in the model cell and rerun; the English checkpoints may still transfer, but they are not the recommended multilingual choice.

In [ ]:
batch_texts = [
    'Apple CEO Tim Cook announced Vision Pro updates in Cupertino.',
    'Maya Chen joined Northstar Analytics in Singapore.',
    'Acme Labs opened an office in Toronto.',
]
batch_results = model.batch_extract_entities(
    batch_texts,
    ['person', 'organization', 'product', 'location'],
    include_spans=True,
    batch_size=3,
)
print(json.dumps(batch_results, indent=2))

spanish = (
    'María González trabaja para Banco del Sol en Madrid desde el 12 de marzo de 2024.'
)
if MODEL_ID != 'fastino/gliner2.5-multi-v1':
    print('\nTip: select gliner2.5-multi-v1 for multilingual production use.\n')
multilingual = model.extract_entities(
    spanish,
    ['person', 'organization', 'location', 'date'],
    include_spans=True,
    include_confidence=True,
)
print(json.dumps(multilingual, indent=2, ensure_ascii=False))

## Recap and production notes

```python
model.extract_entities(text, labels)            # one-window NER
model.extract_entities_long(text, labels)       # chunked full document
model.extract_json(text, record_schema)          # structured records
model.extract(text, schema_with_attributes)      # qualified spans
Classifier(model).classify(text, schema)         # constrained labels
JointIE(model).extract(text, graph_schema)        # typed graph
```

Things worth keeping explicit:

- `small` minimizes CPU latency; `base` is the default English quality choice; `multi` is for multilingual inputs.
- Boundary prediction removes the fixed-width span grid, not the encoded-window limit. A span must fit in one chunk.
- Long-document relations are intra-chunk; increase overlap when both endpoints often straddle boundaries.
- Always inspect `result.feasible` for constrained classification and joint IE.
- Keep `include_spans=True` while validating a schema and assert that each offset slices back to the returned text.
- Zero-shot outputs still require threshold and schema tuning on labeled examples from your domain.

For a repeatable local comparison of all three variants, run:

```bash
python benchmarks/benchmark_gliner25_cpu.py --warm-calls 3
```

The benchmark records model load, seven warm scenarios, native batch throughput, global-offset checks, constraint feasibility, and typed graph checks. It is deliberately separate from Fastino's published 16-dataset macro-F1 evaluation.